# v32 — 硬い核(全PF>10, per-well Optunaでも救えない52井戸)の解剖
なぜ外れるのかを仮説別にグラフで検証する。**新しい計算はなし**(既存アーティファクトのみ):
refs_cache(GR/typewell/真値) / datum_scan(NCC峰) / unsup_meta(XY/family) / v30 preds(素smoother) / OOFブレンド(prior)。

仮説: ①双峰datumの裏モードcommit ②断層/dip折れ ③typewell表現力不足(oracle適合が低い)
④typewellレンジ外逸脱 ⑤幾何(長さ/既知帯/ドリフト) ⑥孤立(近傍/ファミリー無し)
最終セルでルールベースの「原因ラベル」を各井戸に割当てて集計・CSV保存。


In [ ]:
# ===== ロード(全部既存物) =====
import json, glob, time, warnings
from pathlib import Path
import numpy as np, pandas as pd
import matplotlib
import matplotlib.pyplot as plt
warnings.filterwarnings("ignore")
matplotlib.rcParams["font.family"] = "Meiryo"; matplotlib.rcParams["axes.unicode_minus"] = False

PROJECT = Path(r"C:\Users\kosaka256\Documents\rogii_claude")
V28_ART = PROJECT / "v28" / "artifacts_v28"
V29_ART = PROJECT / "v29" / "artifacts_v29"
V29B    = PROJECT / "v29" / "artifacts_v29b"
V30_ART = PROJECT / "v30" / "artifacts_v30"
V31_ART = PROJECT / "v31" / "artifacts_v31"
ART = PROJECT / "v32" / "artifacts_v32"; ART.mkdir(parents=True, exist_ok=True)

HARD = ["ab112491", "4caa7289", "e32da4ec", "73348914", "b8c49c1a", "7d57c75d", "653612bc", "113011eb", "c9e980e8", "708caea9", "fb0904bd", "ab94878e", "137d1e44", "aee6393a", "5aef5c6c", "b32902c0", "493b5b31", "8cc21f01", "91db7070", "43b06c70", "6a8fa194", "efe96181", "d011f41b", "028d7b28", "1d3fbf02", "48c1f673", "70925e23", "00e12e8b", "55efee7f", "4ed93db6", "86454a6f", "9314ff13", "a0e92ed6", "940e709a", "5f75f796", "f8afa78a", "9283ae69", "49dbc14c", "b85f7b06", "ee4aecac", "b0d42b0d", "a2e8e7f6", "9d4272e3", "e7818f7a", "1b6ba517", "8612b37f", "6d590e26", "a8ed028a", "8995c945", "c1708b88", "9e3e364d", "54753541"]

_z = np.load(V31_ART / "refs_cache.npz", allow_pickle=True)
WD_ = {w: _z[w].item() for w in _z.files}
WELLS = sorted(WD_)
HARD = [w for w in HARD if w in WD_]
REST = [w for w in WELLS if w not in set(HARD)]
print(f"wells {len(WELLS)} | hard {len(HARD)} | rest {len(REST)}")

zds = np.load(V31_ART / "datum_scan.npz", allow_pickle=True)
MODES = json.loads(str(zds["meta"]))
META = json.loads((V31_ART / "unsup_meta.json").read_text())
FAM = json.loads((V31_ART / "tw_families.json").read_text())
fam_of = {}
for h, ws in FAM.items():
    for w in ws: fam_of[w] = h

zsm = np.load(V30_ART / "preds_signone.npz", allow_pickle=True)
SM = {w: zsm[f"s_{w}"] for w in WELLS if f"s_{w}" in zsm.files}   # 素smoother予測

# prior(= v28GBM × v29NN combo blend の OOF, v30 と同一構成)
import pyarrow.parquet as pq
tbl = pq.read_table(V28_ART / "data" / "train.parquet",
                    columns=["well", "last_known_tvt", "md_since", "pf_ancc"])
g_p = tbl.column("well").to_pandas().to_numpy()
lk_p = tbl.column("last_known_tvt").to_numpy(zero_copy_only=False).astype(np.float64)
mds_p = tbl.column("md_since").to_numpy(zero_copy_only=False).astype(np.float64)
pfa_p = tbl.column("pf_ancc").to_numpy(zero_copy_only=False).astype(np.float64)
gbm = np.load(V29_ART / "gbm_oof.npy")
nnz = np.load(V29B / "nn_oof_combo.npz")
nn_ens = np.mean([nnz[k] for k in nnz.files], axis=0)
w_nn = float(json.loads((V29B / "results.json").read_text())["combo"]["blend_w"])
res_b = (1 - w_nn) * gbm + w_nn * nn_ens
pp = json.loads((V28_ART / "pp_params.json").read_text())
d_ = res_b * (1 - pp["w_pf"]) + (pfa_p - lk_p) * pp["w_pf"]
d_ = d_ * (1.0 - np.exp(-np.maximum(mds_p, 0.0) / float(pp["tau"]))) * pp.get("alpha", 1.0)
tvt_pr = lk_p + d_
PRIOR = {}
for w in WELLS:
    rows = np.where(g_p == w)[0]
    o = rows[np.argsort(mds_p[rows], kind="stable")]
    if len(o) == len(WD_[w]["tvt_true"]): PRIOR[w] = tvt_pr[o]
print(f"prior aligned: {len(PRIOR)}/{len(WELLS)}")

def rmse(e):
    e = np.asarray(e, float); e = e[np.isfinite(e)]
    return float(np.sqrt(np.mean(e ** 2))) if e.size else np.nan
per_prior = {w: rmse(PRIOR[w] - WD_[w]["tvt_true"]) for w in PRIOR}
print(f"prior overall={rmse(np.concatenate([PRIOR[w]-WD_[w]['tvt_true'] for w in PRIOR])):.3f}"
      f" | hard平均={np.nanmean([per_prior.get(w,np.nan) for w in HARD]):.2f}"
      f" | rest平均={np.nanmean([per_prior.get(w,np.nan) for w in REST]):.2f}")


In [ ]:
# ===== ① 誤差の形: バイアス支配か? + 事例8井戸 =====
def bias_share(w, pred):
    e = pred - WD_[w]["tvt_true"]; e = e[np.isfinite(e)]
    if len(e) < 10: return np.nan, np.nan
    return float(np.mean(e)), float(np.mean(e) ** 2 / max(np.mean(e ** 2), 1e-9))

BS = {w: bias_share(w, PRIOR[w]) for w in PRIOR}
fig, ax = plt.subplots(1, 3, figsize=(16, 4.2))
for ws, lab, al in [(HARD, "hard(52)", 0.9), (REST, "rest", 0.5)]:
    v = [BS[w][1] for w in ws if w in BS and np.isfinite(BS[w][1])]
    ax[0].hist(v, bins=25, range=(0, 1), alpha=al, density=True, label=lab)
ax[0].set_xlabel("bias寄与率 mean(e)^2 / mse"); ax[0].legend(); ax[0].set_title("① 誤差はバイアス(=一定ズレ)支配か")
for ws, lab, al in [(HARD, "hard", 0.9), (REST, "rest", 0.5)]:
    v = [BS[w][0] for w in ws if w in BS and np.isfinite(BS[w][0])]
    ax[1].hist(np.clip(v, -60, 60), bins=48, alpha=al, density=True, label=lab)
ax[1].set_xlabel("平均誤差(バイアス) [ft]"); ax[1].legend(); ax[1].set_title("バイアスの符号と大きさ(双峰なら±バンドル)")
v_h = [per_prior[w] for w in HARD if w in per_prior]
v_r = [per_prior[w] for w in REST if w in per_prior]
ax[2].hist([np.clip(v_h, 0, 50), np.clip(v_r, 0, 50)], bins=25, density=True, label=["hard", "rest"])
ax[2].set_xlabel("per-well RMSE (prior=blend)"); ax[2].legend(); ax[2].set_title("blendでも hard は外れているか")
plt.tight_layout(); plt.show()

hs = sorted([w for w in HARD if w in PRIOR], key=lambda w: -per_prior[w])[:8]
fig, ax = plt.subplots(2, 4, figsize=(19, 7))
for k, w in enumerate(hs):
    a = ax[k // 4][k % 4]; d = WD_[w]
    a.plot(d["q_md"], d["tvt_true"], "k", lw=1.6, label="true")
    a.plot(d["q_md"], PRIOR[w], "r", lw=1, label=f"blend {per_prior[w]:.0f}")
    if w in SM: a.plot(d["q_md"], SM[w], "c", lw=0.8, alpha=0.8, label="smoother")
    a.invert_yaxis(); a.set_title(w[:8], fontsize=9); a.legend(fontsize=6)
plt.suptitle("hard 上位8井戸: 真値 vs blend vs 素smoother"); plt.tight_layout(); plt.show()


In [ ]:
# ===== ②断層/dip折れ ③oracle GR適合 ④typewellレンジ外 ⑤幾何 ⑥孤立 =====
def diag(w):
    d = WD_[w]
    tvt = d["tvt_true"]; z = d["q_z"].astype(np.float64); md = d["q_md"].astype(np.float64)
    geo = tvt + z; ok = np.isfinite(geo)
    r = dict()
    # ② 断層/急折れ: 真geoの1行あたり変化の最大(>5ft=ジャンプ) と dip変動
    if ok.sum() > 20:
        dg = np.diff(geo[ok]); dm = np.diff(md[ok]); v = dg / np.maximum(dm, 0.5)
        r["max_jump"] = float(np.nanmax(np.abs(dg)))
        r["dip_std"] = float(np.nanstd(pd.Series(v).rolling(25, min_periods=5).mean()))
    else:
        r["max_jump"] = np.nan; r["dip_std"] = np.nan
    # ③ oracle GR適合: 真の位置に置いた typewell GR との相関
    twc = d["tw"]; gt = d["grid_tvt"]
    e = np.interp(tvt, gt, np.nan_to_num(twc, nan=np.nan), left=np.nan, right=np.nan)
    m = np.isfinite(e) & np.isfinite(d["q_gr"])
    r["r_true"] = float(np.corrcoef(d["q_gr"][m], e[m])[0, 1]) if m.sum() > 50 else np.nan
    # ④ typewellレンジ外率(twカーブが実在する範囲の外に真TVTが出る割合)
    fin = np.isfinite(twc)
    lo, hi = gt[fin][0], gt[fin][-1]
    r["out_range"] = float(np.mean((tvt < lo) | (tvt > hi)))
    # ⑤ 幾何
    r["ev_len"] = float(md[-1] - md[0]); r["kn_len"] = float(len(d["pre"]) if d["pre"] is not None else 0)
    r["drift"] = float(np.nanmedian(tvt - d["lk"]))
    # ⑥ 孤立
    xy = np.array([[META[x]["cx"], META[x]["cy"]] for x in WELLS])
    i = WELLS.index(w)
    dd = np.hypot(xy[:, 0] - xy[i, 0], xy[:, 1] - xy[i, 1]); dd[i] = np.inf
    r["nn_dist"] = float(dd.min()); r["in_family"] = int(w in fam_of)
    # ① 補助: NCC第2峰と真datumの整合(裏モード検出)
    m_ = MODES.get(w)
    if m_ and len(m_["mu"]) == 2:
        e1 = abs(m_["mu"][0] - m_["dtrue"]); e2 = abs(m_["mu"][1] - m_["dtrue"])
        r["second_peak_true"] = int(e2 < e1)   # 第2峰の方が真に近い=裏モード気味
        r["peak_gap"] = abs(m_["mu"][0] - m_["mu"][1])
    else:
        r["second_peak_true"] = np.nan; r["peak_gap"] = np.nan
    return r

t0 = time.time()
DIAG = pd.DataFrame({w: diag(w) for w in WELLS}).T
DIAG["hard"] = [w in set(HARD) for w in DIAG.index]
DIAG["rmse_prior"] = [per_prior.get(w, np.nan) for w in DIAG.index]
print(f"diag {len(DIAG)} wells ({time.time()-t0:.0f}s)")

cols = [("r_true", "③ oracle GR適合 r(真位置での相関)"), ("out_range", "④ typewellレンジ外率"),
        ("max_jump", "② 最大ジャンプ [ft/行]"), ("dip_std", "② dip変動 std"),
        ("ev_len", "⑤ 評価ゾーン長 [ft]"), ("kn_len", "⑤ 既知帯行数"),
        ("drift", "⑤ 真ドリフト中央値 [ft]"), ("nn_dist", "⑥ 最近傍距離 [ft]")]
fig, ax = plt.subplots(2, 4, figsize=(19, 8))
for k, (c, title) in enumerate(cols):
    a = ax[k // 4][k % 4]
    vh = DIAG.loc[DIAG["hard"], c].astype(float); vr = DIAG.loc[~DIAG["hard"], c].astype(float)
    if c in ("nn_dist",):
        vh = np.log10(np.clip(vh, 10, None)); vr = np.log10(np.clip(vr, 10, None)); title += " (log10)"
    for v, lab, al in [(vr, "rest", 0.5), (vh, "hard", 0.9)]:
        v = v[np.isfinite(v)]
        a.hist(v, bins=25, alpha=al, density=True, label=lab)
    a.set_title(title, fontsize=10); a.legend(fontsize=8)
plt.tight_layout(); plt.show()
print("hard vs rest の中央値比較:")
print(DIAG.groupby("hard")[[c for c, _ in cols] + ["peak_gap", "second_peak_true"]].median().T.round(3))


In [ ]:
# ===== 原因ラベルの割当(ルールベース)と集計 =====
rest_med = DIAG.loc[~DIAG["hard"]].median(numeric_only=True)
def label(w):
    r = DIAG.loc[w]
    tags = []
    if r["out_range"] > 0.15: tags.append("④tw範囲外")
    if np.isfinite(r["r_true"]) and r["r_true"] < 0.35: tags.append("③tw不適合")
    if r["max_jump"] > 8: tags.append("②断層/急折れ")
    bs = BS.get(w, (np.nan, np.nan))
    if bs[1] > 0.7 and abs(bs[0]) > 8: tags.append("①バイアス(裏モード疑い)")
    if r["dip_std"] > 2 * rest_med["dip_std"]: tags.append("②dip暴れ")
    if not tags: tags.append("その他/複合")
    return tags

LB = {w: label(w) for w in HARD if w in DIAG.index}
from collections import Counter
cnt = Counter(t for tags in LB.values() for t in tags)
fig, ax = plt.subplots(figsize=(8, 4))
ks = sorted(cnt, key=lambda k: -cnt[k])
ax.barh(range(len(ks)), [cnt[k] for k in ks])
ax.set_yticks(range(len(ks))); ax.set_yticklabels(ks); ax.invert_yaxis()
ax.set_title(f"硬い核 {len(LB)} 井戸の原因ラベル(重複あり)")
for i, k in enumerate(ks): ax.text(cnt[k] + 0.2, i, str(cnt[k]), va="center")
plt.tight_layout(); plt.show()

out = DIAG.loc[HARD].copy()
out["labels"] = [";".join(LB.get(w, [])) for w in out.index]
out["bias"] = [BS.get(w, (np.nan,))[0] for w in out.index]
out.to_csv(ART / "hardcore_diag.csv", encoding="utf-8-sig")
print("保存:", ART / "hardcore_diag.csv")
print("\n各井戸のラベル:")
for w, tags in sorted(LB.items(), key=lambda kv: -per_prior.get(kv[0], 0)):
    print(f"  {w}  rmse={per_prior.get(w, float('nan')):5.1f}  {';'.join(tags)}")


In [ ]:
# ===== deep-dive: 井戸1本を系列で解剖(WELL を書き換えて再実行) =====
WELL = HARD[0]   # ★ここを見たい井戸IDに(上のラベル一覧からコピー)

d = WD_[WELL]; tvt = d["tvt_true"]; md = d["q_md"]; z = d["q_z"].astype(float)
gt = d["grid_tvt"]; twc = np.nan_to_num(d["tw"], nan=np.nan)
gr = d["q_gr"].astype(float)
e_true = np.interp(tvt, gt, twc, left=np.nan, right=np.nan)     # 真位置のtypewell期待GR
pr = PRIOR.get(WELL); sm = SM.get(WELL)
e_pred = np.interp(pr, gt, twc, left=np.nan, right=np.nan) if pr is not None else None

fig = plt.figure(figsize=(19, 11))
gs = fig.add_gridspec(3, 3)
# (1) 軌跡: true vs blend vs smoother
a = fig.add_subplot(gs[0, :2])
a.plot(md, tvt, "k", lw=1.8, label="true")
if pr is not None: a.plot(md, pr, "r", lw=1.1, label=f"blend rmse={per_prior.get(WELL, float('nan')):.1f}")
if sm is not None: a.plot(md, sm, "c", lw=0.9, alpha=0.8, label="素smoother")
fin = np.isfinite(twc)
a.axhline(gt[fin][0], c="orange", ls=":", label="twレンジ")
a.axhline(gt[fin][-1], c="orange", ls=":")
a.invert_yaxis(); a.legend(fontsize=8); a.set_title(f"{WELL} 軌跡(TVT)"); a.set_xlabel("MD")
# (2) geo_z = TVT+Z(断層/dip折れの検出)
a = fig.add_subplot(gs[0, 2])
a.plot(md, tvt + z, "k", lw=1.2)
a.set_title("真の geo_z=TVT+Z(折れ/ジャンプ確認)"); a.set_xlabel("MD"); a.invert_yaxis()
# (3) GR系列: 観測 vs 真位置期待 vs 予測位置期待
a = fig.add_subplot(gs[1, :])
a.plot(md, gr, "k", lw=0.7, label="観測GR")
a.plot(md, e_true, "g", lw=0.9, alpha=0.85, label="typewell(真位置)")
if e_pred is not None: a.plot(md, e_pred, "r", lw=0.7, alpha=0.7, label="typewell(blend位置)")
a.legend(fontsize=8); a.set_title("GR系列: 真位置に置いてもtypewellと合うか(③の直視)"); a.set_xlabel("MD")
# (4) oracle適合の散布図
a = fig.add_subplot(gs[2, 0])
m = np.isfinite(gr) & np.isfinite(e_true)
a.scatter(e_true[m], gr[m], s=4, alpha=0.3)
rr = np.corrcoef(e_true[m], gr[m])[0, 1] if m.sum() > 10 else np.nan
a.set_xlabel("typewell GR(真位置)"); a.set_ylabel("観測GR"); a.set_title(f"oracle適合散布 r={rr:.2f}")
# (5) NCC走査(datum)
a = fig.add_subplot(gs[2, 1])
if f"c_{WELL}" in zds.files:
    DELTAS = np.arange(-40, 40.001, 0.5)
    a.plot(DELTAS, zds[f"c_{WELL}"])
    mm = MODES.get(WELL, {})
    for mu in mm.get("mu", []): a.axvline(mu, c="r", ls=":")
    a.axvline(mm.get("dtrue", 0), c="g", lw=2, alpha=0.6)
    a.set_title("NCC走査(緑=真datum, 赤=峰)"); a.set_xlabel("δ [ft]")
# (6) 誤差系列
a = fig.add_subplot(gs[2, 2])
if pr is not None: a.plot(md, pr - tvt, "r", lw=0.8, label="blend誤差")
if sm is not None: a.plot(md, sm - tvt, "c", lw=0.7, alpha=0.8, label="smoother誤差")
a.axhline(0, c="k", lw=0.5); a.legend(fontsize=8); a.set_title("誤差系列(一定=バイアス/増加=dip)")
plt.tight_layout(); plt.show()
print(WELL, "| labels:", ";".join(LB.get(WELL, ["(hardではない)"])),
      "| diag:", DIAG.loc[WELL, ["r_true", "out_range", "max_jump", "nn_dist"]].round(3).to_dict())
